# Execute Job Steps #

This sample will create a new Workflow item and some sample jobs that will be used to showcase how to run, stop and finish active current steps. 

### Make connections ###

In [15]:
import arcgis
import re
import datetime
from arcgis.gis.workflowmanager import WorkflowManager
import time

gis = arcgis.gis.GIS(url='https://organizationUrl/portal', username='admin', password='...')
item = gis.content.search('title:"Python Sample"')[0]
workflowManager = WorkflowManager(item)
print("Created connection to workflow manager item")

Connected
Created connection to workflow manager item


### Create a Job ###

In [38]:
# Create Job 
job_templates = workflowManager.job_templates
job_template = {}
for x in job_templates:
    if x.job_template_name == 'Introduction to Workflow Manager':
        job_template = x

jobs = workflowManager.jobs.create(template=job_template.job_template_id,
                                    count=1,
                                    name='Test New Job123',
                                    start='2020-04-02T13:25:50Z',
                                    end='2020-04-02T13:25:50Z',
                                    priority='High',
                                    description='job description',
                                    owner='admin',
                                    assigned='admin',
                                    complete=42,
                                    notes='testing notes'
                                    )
job_one_id = jobs[0]
print('Id: ' + job_one_id)


Id: 6UEEsmk-T_SDkWQJq6-MxA


### Run the current step in the job

In [39]:
job_one = workflowManager.jobs.get(job_one_id)
job_one_diagram = workflowManager.jobs.diagram(job_one_id)
job_one_step_id = job_one_diagram.initial_step_id

# If no step ids are provided, run() will run the current active steps.
job_exec = job_one.run(step_ids=[job_one_step_id])

while not job_exec.done():
    print(f'Progress = {job_exec.status}')
    print(f'{job_exec.messages}\n')
    time.sleep(5)

print(f'Status = {job_exec.status} \n')
print(f'Time elapsed {job_exec.elapse_time}')
print(f'Messages received: \n')
for m in job_exec.messages:
    print(f'{m.message} \n')

# Result() returns the last message received. This will inform you of the final state from running thestep.
print(f'Result = {job_exec.result()}\n')

Progress = Running
[481427599867000: MessageType.STEPSTARTED - {'jobId': '6UEEsmk-T_SDkWQJq6-MxA', 'stepIds': ['df4c8d20-5c99-457f-0be1-21fa8f830760'], 'autoRun': False}]

Status = Complete 

Time elapsed 0:00:00.042007
Messages received: 

{'jobId': '6UEEsmk-T_SDkWQJq6-MxA', 'stepIds': ['df4c8d20-5c99-457f-0be1-21fa8f830760'], 'autoRun': False} 

{'jobId': '6UEEsmk-T_SDkWQJq6-MxA', 'stepId': 'df4c8d20-5c99-457f-0be1-21fa8f830760', 'msgCode': 'StepRunning', 'msg': 'Running...', 'allowedActions': 2, 'helpText': 'To finish this step, click Proceed. If you want to take a break and run it again later, click Pause.', 'userPrompt': 'Welcome to ArcGIS Workflow Manager. This workflow will guide you through some of the basic step types and options for how to use them. This one is a manual step.'} 

Result = 481427645419100: MessageType.STEPINFOREQUIRED - {'jobId': '6UEEsmk-T_SDkWQJq6-MxA', 'stepId': 'df4c8d20-5c99-457f-0be1-21fa8f830760', 'msgCode': 'StepRunning', 'msg': 'Running...', 'allowedA

### Stop the current step

In [40]:
# We can see that the current step is running, but requires user input, lets stop the step. 
stop_exec = job_one.stop()
result = stop_exec.result()

print(f'Status = {stop_exec.status} \n')
print(f'Time elapsed {stop_exec.elapse_time}')
print(f'Messages received: \n')
for m in stop_exec.messages:
    print(f'{m.message} \n')

print(f'Result = {stop_exec.result()}\n')
# The result shows us that the next step was automatically started, as configured in the workflow diagram.

Status = Complete 

Time elapsed 0:00:00.020000
Messages received: 

{'jobId': '6UEEsmk-T_SDkWQJq6-MxA', 'stepIds': ['df4c8d20-5c99-457f-0be1-21fa8f830760']} 

{'jobId': '6UEEsmk-T_SDkWQJq6-MxA', 'stepIds': ['df4c8d20-5c99-457f-0be1-21fa8f830760']} 

Result = 481437613258500: MessageType.STEPPAUSED - {'jobId': '6UEEsmk-T_SDkWQJq6-MxA', 'stepIds': ['df4c8d20-5c99-457f-0be1-21fa8f830760']}



### Finish the current step

In [41]:
# We can see that the current step was paused, and therefore can be finished.
finish_exec = job_one.finish()
result = finish_exec.result()

print(f'Status = {finish_exec.status} \n')
print(f'Time elapsed {finish_exec.elapse_time}')
print(f'Messages received: \n')
for m in finish_exec.messages:
    print(f'{m.message} \n')

print(f'Result = {finish_exec.result()}\n')

Status = Complete 

Time elapsed -1 day, 23:59:59.999000
Messages received: 

{'jobId': '6UEEsmk-T_SDkWQJq6-MxA', 'stepIds': ['df4c8d20-5c99-457f-0be1-21fa8f830760'], 'currentSteps': [{'stepId': '324e7883-8a4b-855e-4319-31072c5f2c37', 'assignedTo': 'admin', 'assignedType': 'User', 'stepName': 'Skippable Step Info', 'canSkip': False}], 'jobStatus': 'Ready'} 

Result = 481442395704400: MessageType.STEPFINISHED - {'jobId': '6UEEsmk-T_SDkWQJq6-MxA', 'stepIds': ['df4c8d20-5c99-457f-0be1-21fa8f830760'], 'currentSteps': [{'stepId': '324e7883-8a4b-855e-4319-31072c5f2c37', 'assignedTo': 'admin', 'assignedType': 'User', 'stepName': 'Skippable Step Info', 'canSkip': False}], 'jobStatus': 'Ready'}

